In [2]:
import sys
!{sys.executable} -m pip install opencv-python matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\ROG\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [3]:
import os
import csv
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# =========================
# 1. 路径设置
# =========================
IMAGE_FOLDER = "artist_images"
SAM_CHECKPOINT = "sam_vit_b_01ec64.pth"
MODEL_TYPE = "vit_b"

OUTPUT_DIR = "analysis_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# 2. 基础检查
# =========================
print("Current working dir:", os.getcwd())
print("Files in current dir:", os.listdir("."))

if not os.path.exists(SAM_CHECKPOINT):
    raise FileNotFoundError(f"找不到模型文件: {SAM_CHECKPOINT}")

if not os.path.exists(IMAGE_FOLDER):
    raise FileNotFoundError(f"找不到图片文件夹: {IMAGE_FOLDER}")

# =========================
# 3. 加载 SAM 大模型
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

sam = sam_model_registry[MODEL_TYPE](checkpoint=SAM_CHECKPOINT)
sam.to(device=device)

mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,
    pred_iou_thresh=0.90,
    stability_score_thresh=0.95,
    min_mask_region_area=800
)

print("SAM loaded successfully.")

# =========================
# 4. 红色检测函数（HSV）
# =========================
def detect_red_mask(rgb_img):
    hsv = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2HSV)

    # 红色在 HSV 中分两段
    lower1 = np.array([0, 80, 80])
    upper1 = np.array([10, 255, 255])

    lower2 = np.array([170, 80, 80])
    upper2 = np.array([180, 255, 255])

    mask1 = cv2.inRange(hsv, lower1, upper1)
    mask2 = cv2.inRange(hsv, lower2, upper2)

    red_mask = (mask1 > 0) | (mask2 > 0)
    return red_mask

# =========================
# 5. 主分析函数
# =========================
def analyse_one_image(image_path, save_index):
    bgr = cv2.imread(image_path)
    if bgr is None:
        print("无法读取图片：", image_path)
        return None

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # SAM 生成 masks
    masks = mask_generator.generate(rgb)

    red_global = detect_red_mask(rgb)

    selected_masks = []

    for m in masks:
        seg = m["segmentation"]   # bool mask
        area = seg.sum()

        if area == 0:
            continue

        red_inside = red_global & seg
        red_ratio = red_inside.sum() / area

        # 条件：区域里有一定比例是红色，并且区域不要太大
        area_ratio = area / (rgb.shape[0] * rgb.shape[1])

        if red_ratio > 0.05 and area_ratio < 0.25:
            selected_masks.append(seg)

    # 合并红色相关区域
    red_region = np.zeros(rgb.shape[:2], dtype=bool)
    for seg in selected_masks:
        red_region |= seg

    # 生成边缘线框
    red_uint8 = (red_region.astype(np.uint8) * 255)
    edges = cv2.Canny(red_uint8, 50, 150)

    # 叠加显示图
    overlay = rgb.copy()
    overlay[red_region] = [255, 0, 0]   # 红色高亮

    # 自动文字说明
    red_pixel_ratio_total = red_region.sum() / (rgb.shape[0] * rgb.shape[1])

    explanation = (
        f"This image was segmented using SAM (vit_h).\n\n"
        f"Red regions were filtered based on HSV colour thresholds and mask overlap.\n\n"
        f"Detected red-thread area ratio: {red_pixel_ratio_total:.4f}\n\n"
        f"The edge view highlights the linear structure of the red thread-like elements,\n"
        f"which may support material and compositional analysis in design research."
    )

    # 画图：原图 / 红色区域 / 线框 / 文字
    fig = plt.figure(figsize=(16, 8))

    ax1 = plt.subplot(2, 2, 1)
    ax1.imshow(rgb)
    ax1.set_title("Original Image")
    ax1.axis("off")

    ax2 = plt.subplot(2, 2, 2)
    ax2.imshow(overlay)
    ax2.set_title("Detected Red Thread Region")
    ax2.axis("off")

    ax3 = plt.subplot(2, 2, 3)
    ax3.imshow(edges, cmap="gray")
    ax3.set_title("Wireframe / Edge Structure")
    ax3.axis("off")

    ax4 = plt.subplot(2, 2, 4)
    ax4.axis("off")
    ax4.text(
        0.02, 0.98,
        explanation,
        va="top",
        wrap=True,
        fontsize=11
    )

    save_path = os.path.join(OUTPUT_DIR, f"analysis_{save_index}.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

    return {
        "image_file": os.path.basename(image_path),
        "selected_mask_count": len(selected_masks),
        "red_area_ratio": round(float(red_pixel_ratio_total), 4),
        "output_file": save_path
    }

# =========================
# 6. 批量处理图片
# =========================
image_files = [
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
]

print("Found images:", len(image_files))

results = []

for i, img_name in enumerate(image_files):
    img_path = os.path.join(IMAGE_FOLDER, img_name)
    print("Processing:", img_name)

    result = analyse_one_image(img_path, i)

    if result is not None:
        results.append(result)
        print("Saved:", result["output_file"])
    else:
        print("Skipped:", img_name)

# =========================
# 7. 保存 CSV
# =========================
csv_path = os.path.join(OUTPUT_DIR, "analysis_results.csv")

with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["image_file", "selected_mask_count", "red_area_ratio", "output_file"]
    )
    writer.writeheader()
    writer.writerows(results)

print("Analysis complete.")
print("CSV saved to:", csv_path)
print("Output folder:", OUTPUT_DIR)

Current working dir: C:\Users\ROG
Files in current dir: ['.arduinoIDE', '.cache', '.cisco', '.idlerc', '.ipynb_checkpoints', '.ipython', '.jupyter', '.matplotlib', '.ms-ad', '.rhinocode', '.thumbnails', '2.9workshop.ipynb', 'analysis_outputs', 'ansel', 'AppData', 'Apple', 'Application Data', 'artist_images', 'Autodesk', 'blenderkit_data', 'books.csv', 'Contacts', 'Cookies', 'Desktop', 'Documents', 'Favorites', 'firstclass.ipynb', 'Links', 'Local Settings', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{c2e0e1d8-9ffb-11ef-aced-98bd805c73b6}.TM.blf', 'NTUSER.DAT{c2e0e1d8-9ffb-11ef-aced-98bd805c73b6}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{c2e0e1d8-9ffb-11ef-aced-98bd805c73b6}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'OneDrive', 'PrintHood', 'Recent', 'red_thread_images.csv', 'sam_vit_b_01ec64.pth', 'sam_vit_h_4b8939.pth', 'Saved Games', 'Searches', 'SendTo', 'Templates', 'Untitled.ipynb', 'WPS Cloud Files', 

In [4]:
import os
import csv
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# =========================
# 1. 路径设置
# =========================
IMAGE_FOLDER = "artist_images"
SAM_CHECKPOINT = "sam_vit_b_01ec64.pth"
MODEL_TYPE = "vit_b"

OUTPUT_DIR = "artistic_rope_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# 2. 加载 SAM
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

sam = sam_model_registry[MODEL_TYPE](checkpoint=SAM_CHECKPOINT)
sam.to(device=device)

mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=24,
    pred_iou_thresh=0.88,
    stability_score_thresh=0.92,
    min_mask_region_area=600
)

# =========================
# 3. 红色检测
# =========================
def detect_red_mask(rgb_img):
    hsv = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2HSV)

    lower1 = np.array([0, 80, 80])
    upper1 = np.array([10, 255, 255])

    lower2 = np.array([170, 80, 80])
    upper2 = np.array([180, 255, 255])

    mask1 = cv2.inRange(hsv, lower1, upper1)
    mask2 = cv2.inRange(hsv, lower2, upper2)

    red_mask = (mask1 > 0) | (mask2 > 0)
    return red_mask

# =========================
# 4. 细化骨架（不用 skimage）
# =========================
def morphological_skeleton(binary_img):
    binary = (binary_img.astype(np.uint8) * 255)
    skel = np.zeros(binary.shape, np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))

    while True:
        eroded = cv2.erode(binary, kernel)
        temp = cv2.dilate(eroded, kernel)
        temp = cv2.subtract(binary, temp)
        skel = cv2.bitwise_or(skel, temp)
        binary = eroded.copy()

        if cv2.countNonZero(binary) == 0:
            break

    return skel > 0

# =========================
# 5. 生长效果
# =========================
def create_growth_layers(binary_mask, steps=6):
    layers = []
    current = (binary_mask.astype(np.uint8) * 255)

    for i in range(steps):
        kernel_size = 3 + i * 2
        kernel = np.ones((kernel_size, kernel_size), np.uint8)
        grown = cv2.dilate(current, kernel, iterations=1)
        layers.append(grown > 0)

    return layers

# =========================
# 6. 分析函数
# =========================
def analyse_artistic_rope(image_path, save_index):
    bgr = cv2.imread(image_path)
    if bgr is None:
        print("Cannot read:", image_path)
        return None

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    masks = mask_generator.generate(rgb)
    red_global = detect_red_mask(rgb)

    selected_masks = []

    for m in masks:
        seg = m["segmentation"]
        area = seg.sum()
        if area == 0:
            continue

        red_inside = red_global & seg
        red_ratio = red_inside.sum() / area
        area_ratio = area / (rgb.shape[0] * rgb.shape[1])

        if red_ratio > 0.05 and area_ratio < 0.25:
            selected_masks.append(seg)

    red_region = np.zeros(rgb.shape[:2], dtype=bool)
    for seg in selected_masks:
        red_region |= seg

    if red_region.sum() == 0:
        print("No red rope-like region found:", image_path)
        return None

    # 骨架
    skeleton = morphological_skeleton(red_region)

    # 生长层
    growth_layers = create_growth_layers(skeleton, steps=6)

    # 原图上的高亮
    overlay = rgb.copy()
    overlay[red_region] = [255, 60, 60]

    # 生长图
    growth_canvas = np.zeros((rgb.shape[0], rgb.shape[1], 3), dtype=np.uint8)
    growth_colors = [
        [30, 30, 30],
        [80, 0, 40],
        [120, 0, 80],
        [160, 20, 100],
        [200, 40, 120],
        [255, 80, 150]
    ]

    for i, layer in enumerate(growth_layers):
        growth_canvas[layer] = growth_colors[i]

    # 骨架白线
    growth_canvas[skeleton] = [255, 255, 255]

    # 自动解释
    rope_area_ratio = red_region.sum() / (rgb.shape[0] * rgb.shape[1])
    skeleton_ratio = skeleton.sum() / max(red_region.sum(), 1)

    explanation = (
        f"This image was segmented with SAM and filtered by red colour overlap.\n\n"
        f"The red rope-like area was isolated as a material trace.\n\n"
        f"The skeleton view reduces the rope to its structural pathway,\n"
        f"while the growth image expands this pathway into a speculative living form.\n\n"
        f"Detected rope area ratio: {rope_area_ratio:.4f}\n"
        f"Skeleton density ratio: {skeleton_ratio:.4f}\n\n"
        f"This transformation interprets the thread not only as an object,\n"
        f"but as a spatial system capable of growth, extension, and organisation."
    )

    # 画图
    fig = plt.figure(figsize=(16, 10))

    ax1 = plt.subplot(2, 2, 1)
    ax1.imshow(rgb)
    ax1.set_title("Original Image")
    ax1.axis("off")

    ax2 = plt.subplot(2, 2, 2)
    ax2.imshow(overlay)
    ax2.set_title("Detected Red Rope Region")
    ax2.axis("off")

    ax3 = plt.subplot(2, 2, 3)
    ax3.imshow(growth_canvas)
    ax3.set_title("Skeleton + Growth Transformation")
    ax3.axis("off")

    ax4 = plt.subplot(2, 2, 4)
    ax4.axis("off")
    ax4.text(
        0.02, 0.98,
        explanation,
        va="top",
        wrap=True,
        fontsize=11
    )

    save_path = os.path.join(OUTPUT_DIR, f"artistic_analysis_{save_index}.png")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

    return {
        "image_file": os.path.basename(image_path),
        "rope_area_ratio": round(float(rope_area_ratio), 4),
        "skeleton_density": round(float(skeleton_ratio), 4),
        "output_file": save_path
    }

# =========================
# 7. 批量处理
# =========================
image_files = [
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
]

print("Found images:", len(image_files))

results = []

for i, img_name in enumerate(image_files):
    img_path = os.path.join(IMAGE_FOLDER, img_name)
    print("Processing:", img_name)

    result = analyse_artistic_rope(img_path, i)

    if result is not None:
        results.append(result)
        print("Saved:", result["output_file"])

# =========================
# 8. 保存 CSV
# =========================
csv_path = os.path.join(OUTPUT_DIR, "artistic_analysis_results.csv")

with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["image_file", "rope_area_ratio", "skeleton_density", "output_file"]
    )
    writer.writeheader()
    writer.writerows(results)

print("Done.")
print("Results saved in:", OUTPUT_DIR)

Using device: cpu
Found images: 6
Processing: image_0.jpg
No red rope-like region found: artist_images\image_0.jpg
Processing: image_1.jpg
Saved: artistic_rope_outputs\artistic_analysis_1.png
Processing: image_2.jpg
Saved: artistic_rope_outputs\artistic_analysis_2.png
Processing: image_3.jpg
Saved: artistic_rope_outputs\artistic_analysis_3.png
Processing: image_4.jpg
Saved: artistic_rope_outputs\artistic_analysis_4.png
Processing: image_5.jpg
Saved: artistic_rope_outputs\artistic_analysis_5.png
Done.
Results saved in: artistic_rope_outputs
